# Exercise Sheet 6 - Worked solutions

SVD, conditioning and sparsity. Questions 1, 4 and 5 are class discussion; Questions 2 and 3 are self-study.

Run the cells in order. The data are generated here, so no downloads are needed. The code uses NumPy, SciPy and Matplotlib. Each block handles one step of the calculation. A fixed random seed makes the examples repeatable; timings still depend on the computer and its numerical libraries.

The notebook with outputs contains the figures and sample results. The version without outputs contains the same explanations and code, ready to run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, eye, diags, vstack
from scipy.sparse.linalg import lsqr
from time import perf_counter

# Scientific notation makes very small errors easier to read.
np.set_printoptions(precision=5, suppress=False)


## Question 1: Conditioning and specified error directions

For the full-column-rank matrix in this question, the least-squares recovery is

$$
x=A^+b,
$$

where $A^+$ is the Moore–Penrose pseudoinverse. This is the unique vector minimising $\|Ax-b\|_2$.

### (a) How the matrix changes different coefficient directions

The two columns of $A_\delta$ become almost identical as $\delta$ decreases. The direction $v_-$ changes the two coefficients with opposite signs, so their effects nearly cancel:

$$
A_\delta v_-=\frac1{\sqrt2}(0,-\delta,-\delta)^T,
\qquad \|A_\delta v_-\|_2=\delta.
$$

By contrast, their effects add along $v_+$. A large coefficient change along $v_-$ can therefore produce a small change in readings. The singular-value ratio measures the largest such disparity over all unit directions; the two supplied directions need not be the exact singular vectors.

With `full_matrices=False`, the SVD of this $3\times2$ matrix returns `U` of shape $(3,2)$, `s` of shape $(2,)$ and `Vh` of shape $(2,2)$. The values in `s` are in descending order, and `Vh` is $V^T$.


In [ ]:
x_true = np.array([2., -1.])
v_plus = np.array([1., 1.])/np.sqrt(2)
v_minus = np.array([1., -1.])/np.sqrt(2)
deltas = [1e-1, 1e-3, 1e-6]

for delta in deltas:
    A = np.array([[1., 1.], [1., 1. + delta], [2., 2. + delta]])
    U, s, Vh = np.linalg.svd(A, full_matrices=False)
    print("delta =", delta)
    print("Condition number:", np.linalg.cond(A))
    print("Singular-value ratio:", s[0]/s[-1])
    print("Response along v+:", np.linalg.norm(A @ v_plus))
    print("Response along v-:", np.linalg.norm(A @ v_minus))


As delta gets smaller, the condition number increases and the response along v- decreases. This is the harder coefficient direction to detect.


### (b) Reading-error direction matters

Coefficient directions belong to $\mathbb R^2$; reading errors belong to $\mathbb R^3$. All three supplied reading errors have length $10^{-7}$, but they need not produce the same recovery error.

For a consistent baseline $b=Ax^*$ and a full-column-rank matrix,

$$
\Delta x=A^+\Delta b,\qquad
\frac{\|\Delta x\|_2}{\|x^*\|_2}
\le \kappa_2(A)\frac{\|\Delta b\|_2}{\|b\|_2}.
$$

This is an upper bound, not a prediction that every direction attains it. Here $A^Te_2=0$: this error is perpendicular to both columns of $A$. It changes the part of the readings that the model cannot fit, but not the least-squares coefficients in exact arithmetic.


In [ ]:
reading_directions = [np.array([1., -1., 0.])/np.sqrt(2),
                      np.array([1., 1., -1.])/np.sqrt(3),
                      np.array([1., 0., 1.])/np.sqrt(2)]
q1_errors = []

# The .3e format prints numbers in scientific notation with three decimal places.
print("delta    direction  absolute amplification      e_b         e_x        e_x/e_b")
for delta in deltas:
    A = np.array([[1., 1.], [1., 1. + delta], [2., 2. + delta]])
    b = A @ x_true
    for j in range(3):
        delta_b = 1e-7*reading_directions[j]
        x_hat = np.linalg.lstsq(A, b + delta_b, rcond=None)[0]
        error = np.linalg.norm(x_hat - x_true)
        e_b = np.linalg.norm(delta_b)/np.linalg.norm(b)
        e_x = error/np.linalg.norm(x_true)
        amplification = error/np.linalg.norm(delta_b)
        print(f"{delta:.0e}        {j + 1}         {amplification:.3e} "
              f"       {e_b:.3e}   {e_x:.3e}   {e_x/e_b:.3e}")
        q1_errors.append((delta, j + 1, e_b, e_x))


### (c) Calculate the change using SVD

Writing $A=U\Sigma V^T$, the full-rank least-squares solution is $x=V\Sigma^{-1}U^Tb$. Subtracting the solutions for $b+\Delta b$ and $b$ gives

$$
\Delta x=V\Sigma^{-1}U^T\Delta b
=\sum_{i=1}^{2}\frac{u_i^T\Delta b}{\sigma_i}v_i.
$$

`U.T @ delta_b` first gives the reading-error components along the output singular directions. Dividing those components by `s` gives the corresponding coefficient changes, then `Vh.T` combines the input directions. A small $\sigma_i$ amplifies the absolute component $u_i^T\Delta b$; it has no such effect if that component is zero.

Reduced `U` does not include the direction perpendicular to the column space. For $e_2$ both retained components vanish in exact arithmetic. Very small computed values need not be exactly zero because of rounding.


In [ ]:
q1_svd_checks = []
for delta in deltas:
    A = np.array([[1., 1.], [1., 1. + delta], [2., 2. + delta]])
    b = A @ x_true
    U, s, Vh = np.linalg.svd(A, full_matrices=False)
    baseline = np.linalg.lstsq(A, b, rcond=None)[0]
    print("delta =", delta)
    for j in range(3):
        delta_b = 1e-7*reading_directions[j]
        # Step 1: express the reading error in the output directions.
        components = U.T @ delta_b
        # Step 2: divide each component by its singular value.
        coefficient_changes = components/s
        # Step 3: combine the input directions to recover the change in x.
        change_svd = Vh.T @ coefficient_changes
        x_hat = np.linalg.lstsq(A, b + delta_b, rcond=None)[0]
        change_solve = x_hat - baseline
        agrees = np.allclose(change_svd, change_solve, rtol=1e-5, atol=3e-9)
        q1_svd_checks.append(agrees)
        print("  Direction", j + 1, "components:", components)
        print("  Change in x:", change_svd, "agrees:", agrees)


### (d) Changing the baseline magnitude

For the same $A$ and $\Delta b$, the exact absolute change $A^+\Delta b$ is unchanged. Multiplying $x^*$ and $b$ by 1000 divides both relative errors by 1000. Thus their ratio is unchanged too. The relative amplification depends on the baseline through $\|b\|_2/\|x^*\|_2$; scaling both equally cancels in that ratio.

The condition number depends only on $A$. The realised relative error also depends on the baseline and the perturbation direction. Small departures from these scaling statements in computed results come from rounding.


In [ ]:
delta = 1e-3
A = np.array([[1., 1.], [1., 1. + delta], [2., 2. + delta]])
delta_b = 1e-7*reading_directions[0]
q1_scaling = []
for scale in [1., 1000.]:
    truth = scale*x_true
    b = A @ truth
    x_hat = np.linalg.lstsq(A, b + delta_b, rcond=None)[0]
    absolute = np.linalg.norm(x_hat - truth)
    relative_b = np.linalg.norm(delta_b)/np.linalg.norm(b)
    relative_x = absolute/np.linalg.norm(truth)
    q1_scaling.append((absolute, relative_b, relative_x))
    print("Scale:", scale)
    print("  Absolute recovery error:", absolute)
    print("  Relative reading error:", relative_b)
    print("  Relative recovery error:", relative_x)


## Question 2: Weighted fitting, SVD and recovery

### (a) Turn weights into an ordinary least-squares problem

The weights multiply squared errors, so the diagonal entries must be their square roots:

$$
S=\operatorname{diag}(\sqrt{w_0},\ldots,\sqrt{w_{40}}),\quad
B=SA,\quad d=Sy_{\rm observed},\qquad
J_w(c)=\|Bc-d\|_2^2.
$$

Using $w_i$ itself as a diagonal entry would give $w_i^2$ in the objective. The three unreliable readings have a noise scale 20 times larger, and their weights are $1/20^2$. Scaling by $S$ therefore makes their noise scales comparable to the others. It does not make the observations exact.


In [ ]:
c_true = np.array([1., 2., -1.])
rng = np.random.default_rng(587)  # Fixed seed: repeatable noise.
z = rng.standard_normal(41)
t_narrow = np.linspace(-0.1, 0.1, 41)
t_wide = np.linspace(-2., 2., 41)
w = np.ones(41)
w[[5, 20, 35]] = 0.0025
noise_scale = np.ones(41)
noise_scale[[5, 20, 35]] = 20.
S = np.diag(np.sqrt(w))

# Each column is one model function evaluated at all input values.
def design(t):
    constant_column = np.ones(len(t))
    return np.column_stack([constant_column, t, np.sin(t)])


Build the four datasets and check the weighting identity. Each trial uses the same weights and noise pattern.


In [ ]:
weight_checks = []
for label, t in [("narrow", t_narrow), ("wide", t_wide)]:
    A = design(t)
    B = S @ A
    for eta in [1e-7, 1e-2]:
        observed = A @ c_true + eta*(noise_scale*z)
        d = S @ observed
        for trial in [np.zeros(3), c_true, np.array([2., -3., 4.])]:
            original = np.sum(w*(A @ trial - observed)**2)
            transformed = np.linalg.norm(B @ trial - d)**2
            weight_checks.append(np.isclose(original, transformed))
print("All objective checks agree:", all(weight_checks))


### (b) Weighting and conditioning answer different questions

For small $t$, $\sin t\approx t-t^3/6$. Thus the last two columns nearly coincide on the narrow range. Increasing $c_1$ and decreasing $c_2$ by the same amount barely changes predictions there: their sum is much better determined than their difference.

Weighting says how to value different observation errors; it does not guarantee a better-conditioned design. Row scaling can improve or worsen the singular-value ratio. Here we check the actual matrices.


In [ ]:
for t in [t_narrow, t_wide]:
    A = design(t)
    B = S @ A
    print("Input interval:", t[0], "to", t[-1])
    print("Singular values of A:", np.linalg.svd(A, compute_uv=False))
    print("Singular values of SA:", np.linalg.svd(B, compute_uv=False))
    print("Condition number of A:", np.linalg.cond(A))
    print("Condition number of SA:", np.linalg.cond(B))


Now plot the two model columns. The narrow-range curves almost overlap.


In [ ]:
for t in [t_narrow, t_wide]:
    fig, ax = plt.subplots()
    ax.plot(t, t, label="t")
    ax.plot(t, np.sin(t), "--", label="sin(t)")
    ax.set(xlabel="t", ylabel="Model function", title=f"Input interval [{t[0]}, {t[-1]}]")
    ax.legend(loc="upper left")
    plt.show()


### (c) Compare three fitting objectives

Ordinary least squares minimises the unweighted residual norm. Weighted least squares minimises $\sqrt{J_w}$ instead, so a slightly larger unweighted residual is not evidence that it failed. The truncated fit minimises the weighted residual only within the span of the retained right singular vectors. It generally does not minimise it over all coefficient vectors.

`lstsq` returns coefficients, a residual-summary array, the retained rank, and singular values. We compute both residual norms explicitly, since that summary can be empty after rank truncation. The retained rank is a numerical choice; the original designs here have rank 3.


In [ ]:
# Repeat the same calculation for two ranges and two noise levels.
q2_results = []
for label, t in [("narrow", t_narrow), ("wide", t_wide)]:
    A = design(t)
    B = S @ A
    for eta in [1e-7, 1e-2]:
        observed = A @ c_true + eta*(noise_scale*z)
        d = S @ observed

        # Three explicit calls show which matrix and data each method uses.
        ordinary = np.linalg.lstsq(A, observed, rcond=None)
        weighted = np.linalg.lstsq(B, d, rcond=None)
        cutoff = np.linalg.lstsq(B, d, rcond=0.01)
        q2_results.append((label, eta, ordinary, weighted, cutoff))


Inspect the fitted coefficients and both residual norms. `result[0]` holds the coefficients and `result[2]` holds the retained rank. The remaining code just prints the comparison table.


In [ ]:
print("range   eta    method       rank    c0       c1       c2       ||r||     sqrt(Jw)   coeff.err")
for label, eta, ordinary, weighted, cutoff in q2_results:
    if label == "narrow":
        t = t_narrow
    else:
        t = t_wide
    A = design(t)
    observed = A @ c_true + eta*(noise_scale*z)
    for method, result in [("unweighted", ordinary), ("weighted", weighted),
                           ("cutoff", cutoff)]:
        c = result[0]
        rank = result[2]
        residual = A @ c - observed
        plain_norm = np.linalg.norm(residual)
        weighted_norm = np.linalg.norm(S @ residual)
        coefficient_error = np.linalg.norm(c - c_true)
        print(f"{label:6} {eta:.0e} {method:10} {rank:3d} "
              f"{c[0]:8.3f} {c[1]:8.3f} {c[2]:8.3f} "
              f"{plain_norm:10.3e} {weighted_norm:10.3e} {coefficient_error:10.3e}")


### (d) Reproduce the singular-value cutoff

For $B=U\Sigma V^T$, keep singular values larger than $0.01\sigma_1$:

$$
c_{\rm cut}=\sum_{i\,\text{retained}}\frac{u_i^Td}{\sigma_i}v_i.
$$

The coefficient component in a discarded direction is set to zero. In the narrow example this direction is approximately the contrast between $c_1$ and $c_2$. Removing it suppresses noise amplification at high noise, but also removes a genuine component of $c_{\rm true}$. At sufficiently low noise, the full fit can recover that component accurately, so the same cutoff is harmful. It trades recovery bias against sensitivity to noise.


In [ ]:
# Work with the narrow-range, high-noise data.
A = design(t_narrow)
observed = A @ c_true + 1e-2*(noise_scale*z)
B = S @ A
d = S @ observed
U, s, Vh = np.linalg.svd(B, full_matrices=False)
keep = s > 0.01*s[0]

# Apply the SVD formula in three separate steps.
components = U[:, keep].T @ d
scaled_components = components/s[keep]
c_cut_svd = Vh[keep, :].T @ scaled_components
expected = np.linalg.lstsq(B, d, rcond=0.01)[0]
print("Singular values:", s)
print("Retained:", keep)
print("Discarded coefficient direction:", Vh[~keep, :])
print("Manual cutoff agrees with lstsq:", np.allclose(c_cut_svd, expected))


Compare the coefficient errors at the two noise levels. The full and truncated fits are recomputed explicitly below.


In [ ]:
for eta in [1e-7, 1e-2]:
    observed = A @ c_true + eta*(noise_scale*z)
    d = S @ observed
    c_full = np.linalg.lstsq(B, d, rcond=None)[0]
    c_cut = np.linalg.lstsq(B, d, rcond=0.01)[0]
    print("Noise scale:", eta)
    print("Full-fit coefficient error:", np.linalg.norm(c_full - c_true))
    print("Cutoff coefficient error:", np.linalg.norm(c_cut - c_true))


### (e) Fit, recovery and prediction

Both test inputs lie outside $[-0.1,0.1]$ but inside $[-2,2]$. The narrow fit therefore extrapolates; the wide fit interpolates. Near the narrow observations, the nearly cancelling columns can give similar curves for very different coefficients. Farther away, $t$ and $\sin t$ separate, exposing the coefficient error.

The cutoff can improve predictions for this noisy example, but does not restore the missing information. Singular values describe the design; a suitable cutoff also depends on the noise, the true coefficient components and the purpose of the prediction. Notice that the singular values are unchanged when only $\eta$ changes.


In [ ]:
test_t = np.array([0.5, 1.5])
A_test = design(test_t)
true_predictions = A_test @ c_true
print("True values at", test_t, ":", true_predictions)
for label, eta, ordinary, weighted, cutoff in q2_results:
    print("Range:", label, "noise scale:", eta)
    for method, result in [("Ordinary", ordinary), ("Weighted", weighted),
                           ("Cutoff", cutoff)]:
        prediction = A_test @ result[0]
        print(" ", method, "prediction:", prediction)
        print("    Error:", prediction - true_predictions)


In [ ]:
# Recompute this one example so the plotted coefficients are explicit.
A = design(t_narrow)
observed = A @ c_true + 1e-2*(noise_scale*z)
B = S @ A
d = S @ observed
c_weighted = np.linalg.lstsq(B, d, rcond=None)[0]
c_cutoff = np.linalg.lstsq(B, d, rcond=0.01)[0]


Use the same fitted coefficients in both figures; only the displayed input interval changes.


In [ ]:
for left, right in [(-0.1, 0.1), (-2., 2.)]:
    plot_t = np.linspace(left, right, 401)
    A_plot = design(plot_t)
    fig, ax = plt.subplots()
    ax.plot(plot_t, A_plot @ c_true, label="True model")
    ax.plot(plot_t, A_plot @ c_weighted, "--", label="Weighted fit")
    ax.plot(plot_t, A_plot @ c_cutoff, ":", label="Truncated fit")
    ax.plot(t_narrow, observed, ".", label="Readings")
    ax.set(xlabel="t", ylabel="y", title=f"Narrow data; view [{left}, {right}]")
    ax.legend(loc="upper left")
    plt.show()


## Question 3: Explore low-rank approximation

### (a) SVD of the image

The noisy image has 80 rows and 100 columns. The reduced factors have shapes $(80,80)$, $(80,)$ and $(80,100)$. Noise makes this example numerically full row rank, even though most visible structure can be represented using few singular terms.


In [ ]:
rng = np.random.default_rng(587)  # Fixed seed: repeatable image noise.
x = np.linspace(-1., 1., 100)
y = np.linspace(-1., 1., 80)
X, Y = np.meshgrid(x, y)
clean = (np.exp(-5*((X + 0.3)**2 + Y**2))
         + 0.7*np.exp(-12*((X - 0.4)**2 + (Y + 0.3)**2)))
image = clean + rng.normal(0., 0.03, clean.shape)
U_image, s_image, Vh_image = np.linalg.svd(image, full_matrices=False)
reconstructed = U_image @ np.diag(s_image) @ Vh_image
print("Shapes:", U_image.shape, s_image.shape, Vh_image.shape)
print("Numerical rank:", np.linalg.matrix_rank(image))
print("Number of singular values:", len(s_image))
print("Reconstruction agrees:", np.allclose(image, reconstructed))

fig, ax = plt.subplots()
ax.plot(np.arange(1, len(s_image) + 1), s_image, ".-")
ax.set(xlabel="Singular-value index i", ylabel="sigma_i", title="Image singular values")
plt.show()


### (b) Retain the first k terms

Each retained singular term contributes one rank-one matrix. Using the same colour limits across all images makes brightness comparable. Increasing $k$ retains more of the original noisy matrix, including its noise; it does not necessarily give a cleaner image.

`plt.subplots(2, 3)` makes two rows and three columns of panels. `axs.flat` visits those six axes one at a time.


In [ ]:
image_ranks = [1, 3, 8, 20, 50]
image_approximations = {}
for k in image_ranks:
    # Select the three factors before multiplying them.
    U_k = U_image[:, :k]
    Sigma_k = np.diag(s_image[:k])
    Vh_k = Vh_image[:k, :]
    image_approximations[k] = U_k @ Sigma_k @ Vh_k


Plot the original and five approximations using a common grey scale.


In [ ]:
# Build the six panels in the same order as their titles.
images = [image]
titles = ["Original noisy image"]
for k in image_ranks:
    images.append(image_approximations[k])
    titles.append(f"k = {k}")

fig, axs = plt.subplots(2, 3, figsize=(11, 7))
for ax, title, values in zip(axs.flat, titles, images):
    ax.imshow(values, cmap="gray", vmin=image.min(), vmax=image.max())
    ax.set(title=title, xlabel="Column", ylabel="Row")
plt.show()


### (c) Error from the discarded singular values

The best rank-$k$ approximation theorem gives

$$
\|A-A_k\|_2=\sigma_{k+1},\qquad
\|A-A_k\|_F^2=\sum_{i>k}\sigma_i^2.
$$

Python indexing starts at zero, so `s_image[k]` is $\sigma_{k+1}$. When $k=80$ the discarded tail is empty and its norm is zero. The following search includes this case, so a qualifying rank always exists for this tolerance.


In [ ]:
image_error_checks = []
print("k      spectral error     next sigma     Frobenius error     tail norm")
for k in image_ranks:
    approximation = image_approximations[k]
    spectral = np.linalg.norm(image - approximation, 2)
    frobenius = np.linalg.norm(image - approximation, "fro")
    tail = np.linalg.norm(s_image[k:])
    image_error_checks.append(np.isclose(spectral, s_image[k])
                              and np.isclose(frobenius, tail))
    print(f"{k:2d} {spectral:18.6f} {s_image[k]:14.6f} "
          f"{frobenius:18.6f} {tail:14.6f}")


Search ranks in increasing order and stop at the first one meeting the error target. The empty singular-value tail at full rank has norm zero.


In [ ]:
all_k = np.arange(1, len(s_image) + 1)
relative_tail = []
for k in all_k:
    error = np.linalg.norm(s_image[k:])/np.linalg.norm(s_image)
    relative_tail.append(error)
for k, error in zip(all_k, relative_tail):
    if error <= 0.10:
        minimum_k = k
        break
print("Smallest k for relative Frobenius error <= 0.10:", minimum_k)
fig, ax = plt.subplots()
ax.plot(all_k, relative_tail)
ax.axhline(0.10, linestyle="--", label="Target 0.10")
ax.set(xlabel="Retained rank k", ylabel="Relative Frobenius error")
ax.legend()
plt.show()


### (d) Storage

Store $U_k$ with $mk$ values, the diagonal of $\Sigma_k$ with $k$ values, and $V_k^T$ with $kn$ values. The total is $k(m+n+1)$. There is no need to store a full $k\times k$ diagonal matrix in a compressed representation, even though `np.diag` is convenient for constructing the demonstration.

Here the original has 8000 values and the factors require $181k$. Among the tested ranks, 1, 3, 8 and 20 save storage; 50 does not. Multiplying out and storing the full $80\times100$ approximation uses 8000 values again. These counts concern matrix entries, not file-format overhead.


In [ ]:
m, n = image.shape
print("Original stored values:", m*n)
for k in image_ranks:
    factor_values = k*(m + n + 1)
    print("k =", k, "factor values =", factor_values,
          "smaller than original:", factor_values < m*n)


### (e) Approximation is not the same as denoising

The theorem minimises distance to noisy $A$, not distance to the unknown clean image. The noisy-data error decreases as $k$ increases. The clean-image error can instead start increasing when added terms primarily reproduce noise. Knowing the simulated truth lets us compare those two goals directly; in a real application we would need other information to choose a denoising rank.


In [ ]:
print("k      distance to noisy image     distance to clean image")
clean_image_errors = {}
for k, approximation in image_approximations.items():
    noisy_error = np.linalg.norm(approximation - image, "fro")
    clean_error = np.linalg.norm(approximation - clean, "fro")
    clean_image_errors[k] = clean_error
    print(f"{k:2d} {noisy_error:26.6f} {clean_error:26.6f}")
print("Best tested rank for the clean image:", min(clean_image_errors, key=clean_image_errors.get))


## Question 4: CSR storage and large least-squares problems

### (a) Encode a small matrix

Read the rows in order. The nonzero counts per row are $1,1,1,2,2$, giving cumulative counts $0,1,2,3,5,7$. These are `indptr`. For example row 3 uses stored positions 3 and 4: its column indices are 0 and 1, with values 1 and 1. A pointer is a position in `data` and `indices`, not a column number.


In [ ]:
C = np.array([[2., 0., 0.],
              [0., 3., 0.],
              [0., 0., 4.],
              [1., 1., 0.],
              [0., 1., 1.]])
data = np.array([2., 3., 4., 1., 1., 1., 1.])
indices = np.array([0, 1, 2, 0, 1, 1, 2])
indptr = np.array([0, 1, 2, 3, 5, 7])
C_csr = csr_matrix((data, indices, indptr), shape=(5, 3))
csr_encode_ok = np.allclose(C_csr.toarray(), C)
print("data:", data)
print("indices:", indices)
print("indptr:", indptr)
print("Agrees with C:", csr_encode_ok)


### (b) Decode a different CSR matrix

The row slices are `0:2`, `2:2`, `2:4` and `4:6`. The second is empty, so row 1 contains only zeros. Repeated pointers describe an empty row without storing any zero values.

The reconstructed matrix is

$$
\begin{bmatrix}
0&4&0&0&-2\\
0&0&0&0&0\\
7&0&3&0&0\\
0&0&5&0&-1
\end{bmatrix}.
$$


In [ ]:
shape = (4, 5)
data = np.array([4., -2., 7., 3., 5., -1.])
indices = np.array([1, 4, 0, 2, 2, 4])
indptr = np.array([0, 2, 2, 4, 6])
rebuilt = np.zeros(shape)
for row in range(shape[0]):
    # Consecutive pointers delimit all stored entries of this row.
    for position in range(indptr[row], indptr[row + 1]):
        column = indices[position]
        rebuilt[row, column] = data[position]

C_other = csr_matrix((data, indices, indptr), shape=shape)
csr_decode_ok = np.allclose(rebuilt, C_other.toarray())
print(rebuilt)
print("Agrees with SciPy:", csr_decode_ok)


### (c) Translate the observation model

A direct reading gives the residual $x_i-d_i$, so its row contains one 1. An average reading gives $\tfrac12x_i+\tfrac12x_{i+1}-h_i$, so its row contains two halves. Therefore

$$
A=\begin{bmatrix}I_p\\H\end{bmatrix},\qquad
b=\begin{bmatrix}d\\h\end{bmatrix},\qquad
H_{i,i}=H_{i,i+1}=\tfrac12.
$$

There are $2p-1$ rows, $p$ columns and $p+2(p-1)=3p-2$ nonzeros. `diags` places the two diagonals in $H$; `vstack` joins the sparse row blocks. The function below returns the system and known truth for each requested size. Resetting the fixed seed makes each size repeatable.


In [ ]:
def make_sparse_example(p):
    # Generate readings from the mathematical model, not from the matrix.
    positions = np.linspace(0., 1., p)
    truth = np.sin(2*np.pi*positions) + 0.3*np.cos(6*np.pi*positions)
    rng = np.random.default_rng(587)  # Fixed seed.
    d = truth + rng.normal(0., 0.02, p)
    h = 0.5*(truth[:-1] + truth[1:]) + rng.normal(0., 0.02, p - 1)

    # Build the direct and adjacent-average blocks sparsely.
    halves = 0.5*np.ones(p - 1)
    H = diags([halves, halves], [0, 1], shape=(p - 1, p), format="csr")
    A = vstack([eye(p, format="csr"), H], format="csr")
    b = np.concatenate([d, h])
    return A, b, truth, d, h



Use p = 200 to check the model and storage before timing larger systems.


In [ ]:
A_sparse, b, truth, d, h = make_sparse_example(200)
objective_checks = []
for trial in [np.zeros(200), truth]:
    model_objective = (np.sum((trial - d)**2)
                       + np.sum((0.5*(trial[:-1] + trial[1:]) - h)**2))
    matrix_objective = np.linalg.norm(A_sparse @ trial - b)**2
    objective_checks.append(np.isclose(model_objective, matrix_objective))
print("Objective checks:", objective_checks)
print("Shape:", A_sparse.shape, "nonzeros:", A_sparse.nnz)

# Dense conversion is solely for the comparison, and stays outside timing.
A_dense = A_sparse.toarray()
csr_bytes = (A_sparse.data.nbytes + A_sparse.indices.nbytes
             + A_sparse.indptr.nbytes)
print("Dense bytes:", A_dense.nbytes, "CSR bytes:", csr_bytes)


### (d) Compare solving time and accuracy

Use identical $A,b$ for both solvers. Each gets an untimed warm-up followed by three timed solves; only the solver call is timed. Report the median to reduce the effect of a single slow measurement. This still describes this run on this computer, not a universal performance guarantee.

LSQR returns a tuple: entry 0 is the solution, entry 1 the stopping code and entry 2 the iteration count. For these noisy, overdetermined systems, code 2 means that LSQR has met its approximate least-squares stopping test. Code 1 instead means it has met its approximate equation-solving test. The residual and agreement checks below assess whether the two timed methods produced comparable answers. CSR stores only the nonzeros, whereas this dense comparison stores every entry.


In [ ]:
benchmark_results = []
for p in [200, 600, 1200]:
    A_sparse, b, truth, d, h = make_sparse_example(p)
    A_dense = A_sparse.toarray()
    sparse_bytes = (A_sparse.data.nbytes + A_sparse.indices.nbytes
                    + A_sparse.indptr.nbytes)

    # Run each solver once before timing it.
    lsqr(A_sparse, b, atol=1e-10, btol=1e-10, iter_lim=200)
    np.linalg.lstsq(A_dense, b, rcond=None)
    sparse_times = []
    dense_times = []
    for repeat in range(3):
        start = perf_counter()
        out = lsqr(A_sparse, b, atol=1e-10, btol=1e-10, iter_lim=200)
        sparse_times.append(perf_counter() - start)

        start = perf_counter()
        dense_result = np.linalg.lstsq(A_dense, b, rcond=None)
        dense_times.append(perf_counter() - start)

    # Save the results; inspect and print them in the next cell.
    benchmark_results.append((p, A_sparse, b, A_dense.nbytes, sparse_bytes,
                               out, dense_result[0], sparse_times, dense_times))
    if max(dense_times) > 10:
        print("A dense solve exceeded 10 seconds; stop increasing the size.")
        break


In [ ]:
benchmark_rows = []
benchmark_agreements = []
print("p       CSR bytes  dense bytes    LSQR (s)  lstsq (s)   ||r_sparse||   ||r_dense||")
for p, A, b, dense_bytes, sparse_bytes, out, x_dense, ts, td in benchmark_results:
    x_sparse = out[0]
    sparse_residual = np.linalg.norm(A @ x_sparse - b)
    dense_residual = np.linalg.norm(A @ x_dense - b)
    agrees = np.allclose(x_sparse, x_dense, rtol=1e-6, atol=1e-8)
    sparse_seconds = np.median(ts)
    dense_seconds = np.median(td)
    benchmark_agreements.append(agrees)
    benchmark_rows.append((p, sparse_bytes, dense_bytes, sparse_seconds,
                           dense_seconds, out[1], out[2], sparse_residual, dense_residual))
    print(f"{p:4d} {sparse_bytes:11d} {dense_bytes:12d} "
          f"{sparse_seconds:11.5f} {dense_seconds:10.5f} "
          f"{sparse_residual:14.6e} {dense_residual:14.6e}")
    print("  Stopping code:", out[1], "iterations:", out[2], "agreement:", agrees)


The residual norms should agree closely even though they are not zero: the noisy direct and average readings generally cannot all be satisfied exactly. Allclose checks coefficient agreement with the tolerances given in the question. We count storage in the matrix arrays; neither this count nor these timings measure peak solver memory.


## Question 5: A polygon as a sparse measurement problem

### (a) Build a row for each measurement

The $i$th midpoint reading is $\tfrac12p_i+\tfrac12p_{i+1}$. Its row in $M$ therefore has halves in columns $i$ and $(i+1)\bmod N$. The final row has a 1 in column 0 for the reference reading. Each row is applied independently to both coordinate vectors:

$$
M P^T-Y=\begin{bmatrix}
(\tfrac12(p_0+p_1)-q_0)^T\\
\vdots\\
(\tfrac12(p_{N-1}+p_0)-q_{N-1})^T\\
(p_0-a)^T
\end{bmatrix},\qquad
Y=\begin{bmatrix}Q^T\\a^T\end{bmatrix}.
$$

Squaring and summing all entries is exactly $J(P)$. The measurement matrix acts on the vertex index; the two columns of $Y$ hold the x and y readings.


In [ ]:
N = 20
theta = np.linspace(0., 2*np.pi, N, endpoint=False)
circle = np.array([np.cos(theta), np.sin(theta)])
P_true = np.array([[1.6, 0.4], [0.2, 0.8]]) @ circle
rng = np.random.default_rng(587)  # Fixed seed: repeatable measurements.
Q_clean = 0.5*(P_true + np.roll(P_true, -1, axis=1))
Q = Q_clean + rng.normal(0., 0.003, Q_clean.shape)
a = P_true[:, 0] + rng.normal(0., 0.003, 2)


In [ ]:
# Each midpoint row stores two halves. The last row stores the reference.
values = []
columns = []
indptr = [0]
for i in range(N):
    values.extend([0.5, 0.5])
    columns.extend([i, (i + 1) % N])
    indptr.append(len(values))
values.append(1.)
columns.append(0)
indptr.append(len(values))

M = csr_matrix((values, columns, indptr), shape=(N + 1, N))
Y_readings = np.vstack([Q.T, a])
print("M shape:", M.shape, "Y shape:", Y_readings.shape, "nonzeros:", M.nnz)


Check the two objective expressions on the zero polygon and on the simulated true polygon.


In [ ]:
polygon_objective_checks = []
for trial in [np.zeros_like(P_true), P_true]:
    predicted_midpoints = 0.5*(trial + np.roll(trial, -1, axis=1))
    midpoint_errors = predicted_midpoints - Q
    reference_error = trial[:, 0] - a
    direct = np.sum(midpoint_errors**2) + np.sum(reference_error**2)
    matrix_form = np.linalg.norm(M @ trial.T - Y_readings, "fro")**2
    polygon_objective_checks.append(np.isclose(direct, matrix_form))
print("Objective checks:", polygon_objective_checks)


### (b) Recover x and y separately

The Frobenius objective is the sum of one least-squares objective for x and another for y. Solving the two columns separately therefore minimises the full objective. `np.vstack` puts the recovered x vector in the first row and y vector in the second row.

The measurement residual compares predicted observations with noisy readings; the recovery error compares reconstructed vertices with the known simulated vertices. They are different quantities in different spaces. Small residual does not by itself guarantee small recovery error.


In [ ]:
polygon_out_x = lsqr(M, Y_readings[:, 0], atol=1e-10, btol=1e-10)
polygon_out_y = lsqr(M, Y_readings[:, 1], atol=1e-10, btol=1e-10)
P_hat = np.vstack([polygon_out_x[0], polygon_out_y[0]])
polygon_residual = np.linalg.norm(M @ P_hat.T - Y_readings, "fro")
polygon_error = np.linalg.norm(P_hat - P_true, "fro")
print("Stopping codes:", polygon_out_x[1], polygon_out_y[1])
print("Iterations:", polygon_out_x[2], polygon_out_y[2])
print("sqrt(J):", polygon_residual)
print("Vertex recovery error:", polygon_error)


### (c) Plot the recovered polygon

Repeating the first vertex closes the last edge. Equal axis scaling gives a unit of x the same displayed length as a unit of y, so the shape is not distorted. Solid and dashed lines distinguish the polygons; the legend sits outside them. Simulated truth reveals recovery errors that we could not measure directly with real observations alone.


In [ ]:
closed_true = np.column_stack([P_true, P_true[:, 0]])
closed_hat = np.column_stack([P_hat, P_hat[:, 0]])
fig, ax = plt.subplots()
ax.plot(closed_true[0], closed_true[1], "o-", label="True polygon")
ax.plot(closed_hat[0], closed_hat[1], ".--", label="Recovered polygon")
ax.set(xlabel="x", ylabel="y", title="Recovery from midpoint and reference readings")
ax.set_aspect("equal")
ax.legend(loc="upper left", bbox_to_anchor=(1., 1.))
plt.show()


### (d) What information does the reference add?

For even $N$, alternating vertex changes cancel in every midpoint:

$$
z=(1,-1,1,-1,\ldots)^T,\qquad M_{\rm mid}z=0.
$$

Without the reference, adding any multiple of $z$ to either coordinate vector produces the same midpoint predictions. For this $N=20$ example the midpoint matrix has rank 19, so recovery is not unique. The reference row sees $z_0=1$ and removes that ambiguity. The augmented matrix has rank 20, although its smallest singular value is still relatively small.

A computed condition number for the rank-deficient matrix is often a very large finite number because its computed smallest singular value is close to, rather than exactly, zero. Mathematically its full recovery condition number is infinite.


In [ ]:
M_dense = M.toarray()  # Small matrix: dense SVD is appropriate.
midpoint_matrix = M_dense[:-1, :]
polygon_ranks = []
for matrix in [midpoint_matrix, M_dense]:
    singular_values = np.linalg.svd(matrix, compute_uv=False)
    rank = np.linalg.matrix_rank(matrix)
    polygon_ranks.append(rank)
    print("Matrix shape:", matrix.shape)
    print("Rank:", rank)
    print("Condition number:", np.linalg.cond(matrix))
    print("Singular values:", singular_values)


Verify the missing direction directly: the average of each adjacent pair of alternating signs is zero.


In [ ]:
alternating = np.ones(N)
alternating[1::2] = -1.
print("Midpoint response to alternating change:",
      np.linalg.norm(midpoint_matrix @ alternating))
print("Response after adding reference:", np.linalg.norm(M_dense @ alternating))


### (e) SVD of the shape

The $2\times N$ centred vertex matrix serves a different purpose from $M$. Its two rows are spatial coordinates. Its left singular vectors are perpendicular directions in the plane; applying $U^T$ expresses each vertex in those coordinates. It rotates or reflects the centred shape, preserving distances. The singular values describe the spread along the two directions.

The true vertices were sampled from an ellipse, and the noisy reconstruction is close to one. SVD does not make an arbitrary polygon into an exact ellipse. The signs of singular vectors are not unique; reversing an axis does not change the interpretation.


In [ ]:
# Centre by subtracting the mean vertex from every column.
centroid = P_hat.mean(axis=1, keepdims=True)
P_centred = P_hat - centroid
U_shape, s_shape, Vh_shape = np.linalg.svd(P_centred, full_matrices=False)
rotated = U_shape.T @ P_centred
print("Spatial directions (columns):", U_shape)
print("Singular values:", s_shape)
print("Directions orthonormal:", np.allclose(U_shape.T @ U_shape, np.eye(2)))
print("Coordinate change preserves norm:",
      np.isclose(np.linalg.norm(rotated), np.linalg.norm(P_centred)))


In [ ]:
closed_centred = np.column_stack([P_centred, P_centred[:, 0]])
fig, ax = plt.subplots()
ax.plot(closed_centred[0], closed_centred[1], "o-", label="Centred polygon")
# Plot each unit direction from the origin; their lengths here are both one.
for j in range(2):
    direction = U_shape[:, j]
    ax.plot([0., direction[0]], [0., direction[1]], "--", label=f"u{j + 1}")
ax.set(xlabel="Centred x", ylabel="Centred y", title="Spatial singular directions")
ax.set_aspect("equal")
ax.legend(loc="upper left", bbox_to_anchor=(1., 1.))
plt.show()



Plot the same centred vertices in the new coordinates. Equal scaling preserves their displayed shape.


In [ ]:
closed_rotated = np.column_stack([rotated, rotated[:, 0]])
fig, ax = plt.subplots()
ax.plot(closed_rotated[0], closed_rotated[1], "o-")
ax.set(xlabel="Coordinate along u1", ylabel="Coordinate along u2",
       title="Vertices in singular-direction coordinates")
ax.set_aspect("equal")
plt.show()


The matrix M combines neighbouring vertex positions to predict observations. The shape SVD instead changes spatial coordinates to display the principal directions of the reconstructed polygon.
